In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
from netCDF4 import Dataset

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

#Specific Libraries
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import matplotlib.lines as mlines
from matplotlib.colors import TwoSlopeNorm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, DataManager_Class

sys.path.append(os.path.join(mainDirectory, "Code", "CodeFiles", "Functions", "Classes"))
from DataPlotting_Classes import DataPlotting_Classes
UltimatePlotting = DataPlotting_Classes.UltimatePlotting_Class

#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=6)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Figures", dataName="Figures", dtype="float32",
                                make_dirs=False)

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#general functions
ocean_fraction=1/4
def PlotLandCoast(ax, ocean_fraction=ocean_fraction, linewidth=10,alpha=0.6,correction=0.003):
    xh=ModelData.xh-ModelData.xh[0]
    ax.axvline(x=xh[-1] * ocean_fraction, color='black', linewidth=3, label='Coastline')
    # Add land and sea colors
    seacolor = (0, 89 / 255, 179 / 255)
    landcolor = (139 / 255, 69 / 255, 19 / 255)
    ax.axhline(y=0, xmin=0, xmax=ocean_fraction-correction, color=seacolor, linewidth=linewidth,alpha=alpha)
    ax.axhline(y=0, xmin=ocean_fraction+correction, xmax=1, color=landcolor, linewidth=linewidth,alpha=alpha)

def GetMeanData(variableName, t_level, z_level):
    timeString = ModelData.timeStrings[t_level]

    if variableName == 'theta_v':
        dataVariable = CallVariable(ModelData, DataManager, timeString, variableName)
        dataVariable_zyx = dataVariable[z_level]
        # perturbation: subtract the horizontal (y, x) mean at each level, at this time
        dataVariable_zyx = dataVariable_zyx - np.mean(dataVariable_zyx, axis=(1, 2), keepdims=True)
    else:
        dataVariable = DataManager.GetTimestepData(DataManager.inputDataDirectory, timeString, variableName=variableName)
        dataVariable_zyx = dataVariable[z_level]

    dataVariable_zx_mean = np.mean(dataVariable_zyx, axis=1)   # along-coast (y) mean
    return dataVariable_zx_mean

In [ ]:
################################################################################################################
#CONFIGURATION PLOTTING FUNCTIONS

In [ ]:
def PlotGridStretching():
    zf = ModelData.zf;zhs = ModelData.zh
    zf_diff = np.diff(zf)*1000
    fig = plt.figure(figsize=(10, 4))
    gs = gridspec.GridSpec(1, 2, figure=fig)
    ax0 = fig.add_subplot(gs[0, 0]); ax1 = fig.add_subplot(gs[0, 1]) 
    
    ax0.set_xlabel('Vertical Level Index');ax0.set_ylabel('z (km)')
    ax1.set_xlabel('Δz');ax1.set_ylabel('z (km)')
    ax0.plot(np.arange(len(zf)), zf, color='black');ax0.set_title('z Level Heights (kms)')
    ax1.plot(zf_diff,zhs, color='black');ax1.set_title('Grid Spacing (m)')

    axes=[ax0,ax1]
    SnapLimitsToTicks(axes, dim="x")
    SnapLimitsToTicks(axes, dim="y")
    for ax in axes:
        ax.set_ylim(zf.min(),zf.max());
# PlotGridStretching()

In [ ]:
################################################################################################################
#PLOTTING FUNCTIONS

In [ ]:
def plot_alongcoast_vertical_u(t, ocean_fraction, ax, cax=None, maxlev=20, index_adjust=0,
                               fontScale=1.0, n_levels=20, n_ticks=7,
                               shared=None, showCbar=True, showXLabel=True, showYLabel=True,
                               showCloud=True, cloudThresh=1e-6,
                               title=None, cbarLabel=None, yTickStep=2, titlePad=14):
    [fs] = UltimatePlotting.GetFontStyles(fontScale)

    zh_slice = slice(0, np.where(ModelData.zh <= maxlev + 1)[0][-1] + 1)
    alongcoast_w  = GetMeanData('winterp', t, zh_slice)
    alongcoast_qv = GetMeanData('qv', t, zh_slice)
    alongcoast    = GetMeanData('uinterp', t, zh_slice)
    alongcoast_w[alongcoast_w < 0.25] = 0

    xh = ModelData.xh - ModelData.xh[0]
    yh = ModelData.zh[zh_slice]
    xf = ModelData.xf - ModelData.xf[0]

    if shared is None:
        [ticks, lim, levels] = UltimatePlotting.GenerateTicksAndLim(alongcoast, nTicks=n_ticks, nLevels=n_levels)
    else:
        [ticks, lim, levels] = shared

    xLabel = 'x (km)' if showXLabel else None
    yLabel = 'z (km)' if showYLabel else None
    UltimatePlotting.PlotContour(
        ax=ax, cax=cax, showCbar=showCbar, xData=xh, yData=yh, zData=alongcoast,
        cmap='RdBu_r', levels=levels, norm=MakeNorm(lim), extend='both',
        cbarLabel=cbarLabel, cbarTicks=ticks, title=title,
        xLabel=xLabel, yLabel=yLabel,
        xTicks=np.linspace(xf[0], xf[-1], 9),
        yTicks=np.arange(0, maxlev + 1e-6, yTickStep), yLim=(0, maxlev),
        fontScale=fontScale)

    if title is not None:
        ax.set_title(title, pad=titlePad, fontweight='normal',
                     fontsize=fs["title"]["fontsize"])

    ax.grid(True, linestyle=':', color='black')
    ax.contour(xh, yh, alongcoast_w, linestyles='solid', linewidths=1.5, colors='black', alpha=0.75)
    contour_qv = ax.contour(xh, yh, alongcoast_qv * 1000, linewidths=1, colors='blue', alpha=0.75)
    ax.clabel(contour_qv, inline=True, fontsize=fs["tick"]["fontsize"], fmt='%1.1f', inline_spacing=5)

    if showCloud:
        cloud = GetMeanData('qc', t, zh_slice) + GetMeanData('qi', t, zh_slice)   # kg/kg
        if np.nanmax(cloud) > cloudThresh:
            ax.contour(xh, yh, cloud, levels=[cloudThresh],
                       linestyles='solid', linewidths=1.5, colors='darkblue')

    PlotLandCoast(ax)

def MakeNorm(lim):
    """Zero-centered norm if the range spans zero, otherwise None (plain linear)."""
    return TwoSlopeNorm(vmin=lim[0], vcenter=0, vmax=lim[1]) if lim[0] < 0 < lim[1] else None
from scipy.ndimage import gaussian_filter

def plot_single_horizontal_wind(t, ax, cax, var, c_label, ocean_fraction, index_adjust=0,
                                fontScale=1.0, n_levels=15, n_ticks=7,
                                shared=None, showCbar=True, showXLabel=True, showYLabel=True,
                                yLimit=None, xLimit=None, title=None, titlePad=14,
                                smoothKm=None):
    [fs] = UltimatePlotting.GetFontStyles(fontScale)

    zHeight_m=500
    zlev = np.abs(ModelData.zh*1e3-zHeight_m).argmin()

    field_tzyx = DataManager.GetTimestepData(DataManager.inputDataDirectory,
                                             timeString=ModelData.timeStrings[t], variableName=var)
    field = np.asarray(field_tzyx[zlev], dtype=float)

    xh = ModelData.xh - ModelData.xh[0]
    yh = ModelData.yh - ModelData.yh[0]
    xf = ModelData.xf - ModelData.xf[0]
    yf = ModelData.yf - ModelData.yf[0]

    # smooth on load, before the colorbar range is computed
    if smoothKm is not None and smoothKm > 0:
        dx, dy = np.mean(np.diff(xh)), np.mean(np.diff(yh))        # km per grid point
        field = gaussian_filter(field, sigma=(smoothKm / dy, smoothKm / dx), mode='nearest')

    if shared is None:
        clipped = np.clip(field, np.percentile(field, 1), np.percentile(field, 99))
        [ticks, lim, levels] = UltimatePlotting.GenerateTicksAndLim(clipped, nTicks=n_ticks, nLevels=n_levels)
    else:
        [ticks, lim, levels] = shared

    xLabel = 'x (km)' if showXLabel else None
    yLabel = 'y (km)' if showYLabel else None          # was 'z (km)'
    UltimatePlotting.PlotContour(
        ax=ax, cax=cax, showCbar=showCbar, xData=xh, yData=yh, zData=field,
        cmap='RdBu_r', levels=levels, norm=MakeNorm(lim), extend='both',
        cbarLabel=c_label, cbarTicks=ticks, title=title,
        xLabel=xLabel, yLabel=yLabel,
        xTicks=np.linspace(xf[0], xf[-1], 9), yTicks=np.linspace(yf[0], yf[-1], 9),
        fontScale=fontScale)

    if title is not None:
        ax.set_title(title, pad=titlePad, fontweight='normal',
                     fontsize=fs["title"]["fontsize"])

    ax.axvline(x=xh[-1] * ocean_fraction, color='black', linewidth=3)

    if yLimit is not None:
        ax.set_ylim(yLimit[0], yLimit[1])
        ax.set_yticks(np.arange(yLimit[0], yLimit[1] + 1, yLimit[2]))
    if xLimit is not None:
        ax.set_xlim(xLimit[0], xLimit[1])
        ax.set_xticks(np.arange(xLimit[0], xLimit[1] + 1, xLimit[2]))


def GetSharedURange(t, maxlev=10, n_ticks=7, n_levels=20, zlev=3):
    zh_slice = slice(0, np.where(ModelData.zh <= maxlev + 1)[0][-1] + 1)
    u_vert = GetMeanData('uinterp', t, zh_slice)

    field = DataManager.GetTimestepData(DataManager.inputDataDirectory,
                                        timeString=ModelData.timeStrings[t],
                                        variableName='uinterp')[zlev]
    u_horiz = np.clip(field, np.percentile(field, 1), np.percentile(field, 99))

    # one nice range covering both panels
    return UltimatePlotting.GenerateTicksAndLim([u_vert, u_horiz], nTicks=n_ticks, nLevels=n_levels)
    
def plot_alongcoast_clouds(t, ocean_fraction, ax, cax, maxlev=20, index_adjust=0,
                           fontScale=1.0, n_levels=15, n_ticks=6,
                           showXLabel=True, showYLabel=True,
                           showCloud=True, cloudThresh=1e-6,
                           title=None, cbarLabel=None, yTickStep=2, titlePad=14):
    [fs] = UltimatePlotting.GetFontStyles(fontScale)

    zh_slice = slice(0, np.where(ModelData.zh <= maxlev + 1)[0][-1] + 1)
    alongcoast_w  = GetMeanData('winterp', t, zh_slice)
    alongcoast_qv = GetMeanData('qv', t, zh_slice)
    thetarho_p    = GetMeanData('theta_v', t, zh_slice)     # theta_rho' (K)
    alongcoast_w[alongcoast_w < 0.25] = 0

    xh = ModelData.xh - ModelData.xh[0]
    yh = ModelData.zh[zh_slice]
    xf = ModelData.xf - ModelData.xf[0]

    [ticks, lim, levels] = UltimatePlotting.GenerateTicksAndLim(thetarho_p, nTicks=n_ticks, nLevels=n_levels)

    xLabel = 'x (km)' if showXLabel else None
    yLabel = 'z (km)' if showYLabel else None
    UltimatePlotting.PlotContour(
        ax=ax, cax=cax, xData=xh, yData=yh, zData=thetarho_p,
        cmap='RdBu_r', levels=levels, norm=MakeNorm(lim), extend='both', cbarTicks=ticks,
        cbarLabel=cbarLabel, title=title,
        xLabel=xLabel, yLabel=yLabel,
        xTicks=np.linspace(xf[0], xf[-1], 9), yTicks=np.arange(0, maxlev + 1e-6, yTickStep), yLim=(0, maxlev),
        fontScale=fontScale)

    if title is not None:
        ax.set_title(title, pad=titlePad, fontweight='normal',
                     fontsize=fs["title"]["fontsize"])

    ax.grid(True, linestyle=':', color='black')
    ax.contour(xh, yh, alongcoast_w, linestyles='solid', linewidths=1.5, colors='black', alpha=0.75)
    contour_qv = ax.contour(xh, yh, alongcoast_qv * 1000, linewidths=1, colors='blue', alpha=0.75)
    ax.clabel(contour_qv, inline=True, fontsize=fs["tick"]["fontsize"], fmt='%1.1f', inline_spacing=5)

    if showCloud:
        cloud = GetMeanData('qc', t, zh_slice) + GetMeanData('qi', t, zh_slice)   # kg/kg
        if np.nanmax(cloud) > cloudThresh:
            ax.contour(xh, yh, cloud, levels=[cloudThresh],
                       linestyles='solid', linewidths=1.5, colors='darkblue')

    PlotLandCoast(ax)

In [ ]:
#Make Combined Plot
def MakeCombinedPlot():
    t = np.abs(ModelData.time_hrs-14).argmin()  # 14 LT
    channel_aspect_ratio = 5; figwidth = 25
    fontScale = 2.0
    titlePad = 14

    [fig, axes, caxes] = UltimatePlotting.MakeAlignedGridFigure(
        nRows=2, nCols=2,
        figSize=(figwidth, figwidth / channel_aspect_ratio * 2),
        hasCbar=[[True, True], [False, True]],
        cbarSide="right", rowGapRatio=0.05, colGapRatio=0.08,
        cbarPaddingRatioRight=0.01, cbarThicknessRatio=0.04,
        shareX=True, shareY=False)

    # share y only between the two bottom panels
    axes[1, 1].sharey(axes[1, 0])
    plt.setp(axes[1, 1].get_yticklabels(), visible=False)

    maxlev_left, maxlev_right = 6, 10
    sharedU = GetSharedURange(t, maxlev=maxlev_left, n_ticks=7, n_levels=20)

    # top-left (u, vertical)
    plot_alongcoast_vertical_u(t, ocean_fraction, ax=axes[0, 0], cax=caxes[0, 0], maxlev=maxlev_left,
                               fontScale=fontScale, shared=sharedU, showCbar=True, showXLabel=False,
                               title=r"A) $u$ (m s$^{-1}$)", yTickStep=1, titlePad=titlePad)

    # bottom-left (u, horizontal): restored
    plot_single_horizontal_wind(t, axes[1, 0], None, 'uinterp', None, ocean_fraction,
                                fontScale=fontScale, shared=sharedU, showCbar=False,
                                xLimit=(0, 400, 50), titlePad=titlePad)

    # top-right (theta_rho')
    plot_alongcoast_clouds(t, ocean_fraction, ax=axes[0, 1], cax=caxes[0, 1], maxlev=maxlev_right,
                           fontScale=fontScale, showYLabel=False,
                           title=r"B) $\theta_\rho'$ (K) (top) and $w$ (m s$^{-1}$) (bottom)",
                           yTickStep=2, titlePad=titlePad)

    # bottom-right (w, horizontal)
    yLimit=(80,110,5) if ModelData.simulationNumber in [6] else (75, 125, 10)
    smoothKm=0.8 if ModelData.simulationNumber in [6] else None
    plot_single_horizontal_wind(t, axes[1, 1], caxes[1, 1], 'winterp', None, ocean_fraction,
                                fontScale=fontScale, showYLabel=False,
                                yLimit=yLimit, xLimit=(0, 400, 50), titlePad=titlePad,smoothKm=smoothKm)

    UltimatePlotting.StretchColorbarAcrossAxes(cax=caxes[0, 0],
                                               axesList=[axes[0, 0], axes[1, 0]],
                                               cbarSide="right")

    SaveFigure(fig, plotFileTitle="Initial_ContourPlots")
    return fig

#Plot Saving Function
def SaveFigure(fig,plotFileTitle):
    plotFileName = f"{plotFileTitle}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType="Initial_Figures/Figures")
    print(f"Saving Plot to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
################################################################################################################
#PLOTTING FIGURES

In [ ]:
MakeCombinedPlot()